# Cluster Router Experiment Without Tenure

This notebook tests whether clustering on a no-tenure `abstract_shared` helps cross-domain churn transfer.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import StandardScaler


SCRIPT_DIR = Path(__file__).resolve().parent
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from abstract_shared_ibm_test import (  # noqa: E402
    ROOT,
    RANDOM_STATE,
    _num,
    _rank_order,
    _yes_no,
    compute_metrics,
    fit_imputer,
    fit_xgb,
    load_bank,
    load_cell2cell,
    load_ibm,
    md_table,
    split_domain,
    transform_imputer,
    best_f1_threshold,
)


RESULT_PATH = ROOT / "results" / "cluster_router_no_tenure_results.md"
NOTEBOOK_PATH = ROOT / "notebooks" / "comparison" / "cluster_router_no_tenure.ipynb"


def build_abstract_cell(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    activity = _num(df["MonthlyMinutes"])
    monetary = _num(df["MonthlyRevenue"])
    capacity = _num(df["TotalRecurringCharge"])
    overage = _num(df["OverageMinutes"])
    change = _num(df["PercChangeMinutes"]).abs().fillna(0) + _num(df["PercChangeRevenues"]).abs().fillna(0)
    support = _num(df["RetentionCalls"]).fillna(0) + _num(df["CustomerCareCalls"]).fillna(0) + df["MadeCallToRetentionTeam"].map({"Yes": 1.0, "No": 0.0}).fillna(0)
    relationship = _num(df["UniqueSubs"]).fillna(0) + _num(df["ActiveSubs"]).fillna(0)
    X = pd.DataFrame(
        {
            "age": _num(df["AgeHH1"]),
            "partner_flag": _yes_no(df["MaritalStatus"]),
            "children_flag": _yes_no(df["ChildrenInHH"]),
            "relationship_depth": relationship,
            "activity_volume": activity,
            "monetary_volume": monetary,
            "capacity": capacity,
            "pressure_ratio": overage / (activity + 1.0),
            "change_intensity": change,
            "support_intensity": support,
            "volume_to_capacity": monetary / (capacity + 1.0),
            "activity_to_capacity": activity / (capacity + 1.0),
            "monetary_to_capacity": monetary / (capacity + 1.0),
            "support_to_activity": support / (activity + 1.0),
            "relationship_to_activity": relationship / (activity + 1.0),
        }
    )
    return X, df["Churn"].astype(int)


def build_abstract_bank(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    activity = _num(df["Total_Trans_Ct"])
    monetary = _num(df["Total_Trans_Amt"])
    capacity = _num(df["Credit_Limit"])
    pressure = _num(df["Avg_Utilization_Ratio"])
    change = (_num(df["Total_Ct_Chng_Q4_Q1"]) - 1.0).abs().fillna(0) + (_num(df["Total_Amt_Chng_Q4_Q1"]) - 1.0).abs().fillna(0)
    support = _num(df["Contacts_Count_12_mon"]).fillna(0) + _num(df["Months_Inactive_12_mon"]).fillna(0)
    relationship = _num(df["Total_Relationship_Count"]).fillna(0)
    X = pd.DataFrame(
        {
            "age": _num(df["Customer_Age"]),
            "partner_flag": _rank_order(df["Marital_Status"], {"Married": 1.0, "Single": 0.0, "Divorced": 0.0, "Unknown": np.nan}),
            "children_flag": np.where(_num(df["Dependent_count"]).fillna(0) > 0, 1.0, 0.0),
            "relationship_depth": relationship,
            "activity_volume": activity,
            "monetary_volume": monetary,
            "capacity": capacity,
            "pressure_ratio": pressure,
            "change_intensity": change,
            "support_intensity": support,
            "volume_to_capacity": monetary / (capacity + 1.0),
            "activity_to_capacity": activity / (capacity + 1.0),
            "monetary_to_capacity": monetary / (capacity + 1.0),
            "support_to_activity": support / (activity + 1.0),
            "relationship_to_activity": relationship / (activity + 1.0),
        }
    )
    return X, df["Attrition_Flag"].astype(int)


def build_abstract_ibm(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    activity = _num(df["MonthlyCharges"])
    monetary = _num(df["TotalCharges"])
    contract_map = {"Month-to-month": 1.0, "One year": 2.0, "Two year": 3.0}
    capacity = df["Contract"].map(contract_map).astype(float)
    internet_active = np.where(df["InternetService"].astype(str).eq("No"), 0.0, 1.0)
    service_count = (
        _yes_no(df["PhoneService"]).fillna(0)
        + _yes_no(df["MultipleLines"]).fillna(0)
        + internet_active
        + _yes_no(df["OnlineSecurity"]).fillna(0)
        + _yes_no(df["OnlineBackup"]).fillna(0)
        + _yes_no(df["DeviceProtection"]).fillna(0)
        + _yes_no(df["TechSupport"]).fillna(0)
        + _yes_no(df["StreamingTV"]).fillna(0)
        + _yes_no(df["StreamingMovies"]).fillna(0)
        + _yes_no(df["PaperlessBilling"]).fillna(0)
    )
    support_count = (
        _yes_no(df["OnlineSecurity"]).fillna(0)
        + _yes_no(df["OnlineBackup"]).fillna(0)
        + _yes_no(df["DeviceProtection"]).fillna(0)
        + _yes_no(df["TechSupport"]).fillna(0)
    )
    contract_instability = df["Contract"].map({"Month-to-month": 1.0, "One year": 0.5, "Two year": 0.0}).astype(float)
    change = contract_instability + (activity - monetary / (service_count + 1.0)).abs() / (activity + 1.0)
    X = pd.DataFrame(
        {
            "age": _num(df["SeniorCitizen"]),
            "partner_flag": _yes_no(df["Partner"]),
            "children_flag": _yes_no(df["Dependents"]),
            "relationship_depth": service_count,
            "activity_volume": activity,
            "monetary_volume": monetary,
            "capacity": capacity,
            "pressure_ratio": activity / (monetary + 1.0),
            "change_intensity": change,
            "support_intensity": support_count,
            "volume_to_capacity": activity / (capacity + 1.0),
            "activity_to_capacity": activity / (capacity + 1.0),
            "monetary_to_capacity": monetary / (capacity + 1.0),
            "support_to_activity": support_count / (activity + 1.0),
            "relationship_to_activity": service_count / (activity + 1.0),
        }
    )
    return X, df["Churn"].astype(int)


def make_source_pool(cell_X_train, cell_y_train, bank_X_train, bank_y_train):
    X = pd.concat([cell_X_train, bank_X_train], axis=0, ignore_index=True)
    y = pd.concat([cell_y_train.reset_index(drop=True), bank_y_train.reset_index(drop=True)], axis=0, ignore_index=True)
    domain = pd.Series(
        ["Cell2Cell"] * len(cell_X_train) + ["BankChurners"] * len(bank_X_train),
        name="domain",
    )
    return X, y, domain


def fit_clusterer(X_train: pd.DataFrame, k: int):
    imp, X_train_imp = fit_imputer(X_train)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    kmeans.fit(X_train_scaled)
    train_cluster = pd.Series(kmeans.labels_, index=X_train_imp.index, name="cluster")
    return {
        "imputer": imp,
        "train_imp": X_train_imp,
        "scaler": scaler,
        "kmeans": kmeans,
        "train_cluster": train_cluster,
    }


def predict_cluster(clusterer: dict, X: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X_imp = transform_imputer(clusterer["imputer"], X)
    X_scaled = clusterer["scaler"].transform(X_imp)
    X_imp = X_imp.reset_index(drop=True)
    cluster = pd.Series(clusterer["kmeans"].predict(X_scaled), index=X_imp.index, name="cluster")
    return X_imp, cluster


def add_cluster_one_hot(X_imp: pd.DataFrame, cluster: pd.Series, k: int) -> pd.DataFrame:
    cluster_df = pd.DataFrame(
        {f"cluster_{i}": (cluster.to_numpy() == i).astype(float) for i in range(k)},
        index=X_imp.index,
    )
    return pd.concat([X_imp, cluster_df], axis=1)


def cluster_diagnostics(cluster: pd.Series, domain: pd.Series, churn: pd.Series, X_scaled: np.ndarray) -> dict:
    cluster_np = cluster.to_numpy()
    domain_np = domain.to_numpy()
    churn_np = churn.to_numpy()
    counts = pd.Series(cluster_np).value_counts().sort_index()
    domain_table = pd.crosstab(cluster_np, domain_np)
    churn_table = pd.crosstab(cluster_np, churn_np)
    purity = float(domain_table.max(axis=1).sum() / domain_table.to_numpy().sum())
    churn_purity = float(churn_table.max(axis=1).sum() / churn_table.to_numpy().sum())
    n_clusters = len(counts)
    sample_size = min(5000, len(cluster_np))
    sil = float(silhouette_score(X_scaled, cluster_np, sample_size=sample_size, random_state=RANDOM_STATE))
    return {
        "n_clusters": n_clusters,
        "cluster_min_size": int(counts.min()),
        "cluster_max_size": int(counts.max()),
        "domain_ari": float(adjusted_rand_score(domain_np, cluster_np)),
        "domain_nmi": float(normalized_mutual_info_score(domain_np, cluster_np)),
        "domain_purity": purity,
        "churn_ari": float(adjusted_rand_score(churn_np, cluster_np)),
        "churn_nmi": float(normalized_mutual_info_score(churn_np, cluster_np)),
        "churn_purity": churn_purity,
        "silhouette": sil,
    }


def router_fit(X_train: pd.DataFrame, y_train: pd.Series, cluster_train: pd.Series, min_cluster_size: int = 250):
    global_model = fit_xgb(X_train, y_train)
    models: dict[int, object] = {}
    cluster_sizes: dict[int, int] = {}
    for c in sorted(cluster_train.unique()):
        mask = cluster_train == c
        cluster_sizes[int(c)] = int(mask.sum())
        y_c = y_train.loc[mask]
        if len(y_c) < min_cluster_size or y_c.nunique() < 2:
            continue
        models[int(c)] = fit_xgb(X_train.loc[mask], y_c)
    return {"global": global_model, "models": models, "cluster_sizes": cluster_sizes}


def router_predict(router: dict, X: pd.DataFrame, cluster: pd.Series) -> np.ndarray:
    prob = np.zeros(len(X), dtype=float)
    cluster_np = cluster.to_numpy()
    for c in np.unique(cluster_np):
        mask = cluster_np == c
        model = router["models"].get(int(c), router["global"])
        prob[mask] = model.predict_proba(X.iloc[mask])[:, 1]
    return prob


def evaluate_three_domains(model, cell_test, cell_y_test, bank_test, bank_y_test, ibm_test, ibm_y_test):
    return pd.DataFrame(
        [
            {"split": "Cell2Cell holdout", **compute_metrics(cell_y_test, model.predict_proba(cell_test)[:, 1])},
            {"split": "BankChurners holdout", **compute_metrics(bank_y_test, model.predict_proba(bank_test)[:, 1])},
            {"split": "IBM holdout", **compute_metrics(ibm_y_test, model.predict_proba(ibm_test)[:, 1])},
        ]
    )


def cluster_summary_table(cluster: pd.Series, domain: pd.Series, churn: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame({"cluster": cluster, "domain": domain, "churn": churn})
    counts = pd.crosstab(df["cluster"], df["domain"])
    counts["total"] = counts.sum(axis=1)
    counts["purity"] = counts.drop(columns=["total"]).max(axis=1) / counts["total"]
    counts["churn_rate"] = df.groupby("cluster")["churn"].mean()
    return counts.reset_index().rename(columns={"cluster": "cluster_id"})


def ibm_cluster_summary(cluster: pd.Series, ibm_y: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame({"cluster": cluster, "churn": ibm_y})
    summary = df.groupby("cluster").agg(count=("churn", "size"), churn_rate=("churn", "mean"))
    return summary.reset_index().rename(columns={"cluster": "cluster_id"})


def write_notebook(script_text: str):
    nb = {
        "cells": [
            {
                "cell_type": "markdown",
                "metadata": {},
                "source": [
                    "# Cluster Router Experiment Without Tenure\n",
                    "\n",
                    "This notebook tests whether clustering on a no-tenure `abstract_shared` helps cross-domain churn transfer.\n",
                ],
            },
            {
                "cell_type": "code",
                "execution_count": None,
                "metadata": {},
                "outputs": [],
                "source": script_text.splitlines(keepends=True),
            },
        ],
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.11"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    NOTEBOOK_PATH.write_text(json.dumps(nb, ensure_ascii=False, indent=2), encoding="utf-8")


def main():
    cell_df = load_cell2cell(ROOT / "data" / "raw" / "cell2celltrain.csv")
    bank_df = load_bank(ROOT / "data" / "external" / "credit_card_churn" / "BankChurners.csv")
    ibm_df = load_ibm(ROOT / "data" / "external" / "ibm_telco" / "Telco-Customer-Churn.csv")

    cell_X, cell_y = build_abstract_cell(cell_df)
    bank_X, bank_y = build_abstract_bank(bank_df)
    ibm_X, ibm_y = build_abstract_ibm(ibm_df)

    cX_train, cX_test, cy_train, cy_test = split_domain(cell_X, cell_y)
    bX_train, bX_test, by_train, by_test = split_domain(bank_X, bank_y)
    iX_train, iX_test, iy_train, iy_test = split_domain(ibm_X, ibm_y)

    source_X_train, source_y_train, source_domain_train = make_source_pool(cX_train, cy_train, bX_train, by_train)
    source_X_test = pd.concat([cX_test, bX_test], axis=0, ignore_index=True)
    source_y_test = pd.concat([cy_test.reset_index(drop=True), by_test.reset_index(drop=True)], axis=0, ignore_index=True)
    source_domain_test = pd.Series(["Cell2Cell"] * len(cX_test) + ["BankChurners"] * len(bX_test), name="domain")

    source_imp, source_X_train_imp = fit_imputer(source_X_train)
    source_X_test_imp = transform_imputer(source_imp, source_X_test)
    ibm_X_imp = transform_imputer(source_imp, iX_test).reset_index(drop=True)

    baseline_model = fit_xgb(source_X_train_imp, source_y_train)
    baseline_table = evaluate_three_domains(
        baseline_model,
        transform_imputer(source_imp, cX_test),
        cy_test,
        transform_imputer(source_imp, bX_test),
        by_test,
        ibm_X_imp,
        iy_test,
    )

    baseline_ibm_best = best_f1_threshold(iy_test, baseline_model.predict_proba(ibm_X_imp)[:, 1])

    k_values = list(range(2, 9))
    sweep_rows = []
    augmented_tables = []
    router_tables = []
    cluster_tables = {}

    for k in k_values:
        clusterer = fit_clusterer(source_X_train, k)
        source_train_cluster = clusterer["train_cluster"]
        source_test_imp, source_test_cluster = predict_cluster(clusterer, source_X_test)
        ibm_test_imp, ibm_test_cluster = predict_cluster(clusterer, iX_test)

        source_scaled = clusterer["scaler"].transform(clusterer["train_imp"])
        diag = cluster_diagnostics(source_train_cluster, source_domain_train, source_y_train, source_scaled)

        cluster_tables[k] = {
            "source": cluster_summary_table(source_train_cluster, source_domain_train, source_y_train),
            "ibm": ibm_cluster_summary(ibm_test_cluster, iy_test),
        }

        train_aug = add_cluster_one_hot(source_X_train_imp, source_train_cluster, k)
        cell_test_imp = source_test_imp.iloc[: len(cX_test)].reset_index(drop=True)
        bank_test_imp = source_test_imp.iloc[len(cX_test):].reset_index(drop=True)
        cell_aug = add_cluster_one_hot(cell_test_imp, source_test_cluster.iloc[: len(cX_test)].reset_index(drop=True), k)
        bank_aug = add_cluster_one_hot(bank_test_imp, source_test_cluster.iloc[len(cX_test):].reset_index(drop=True), k)
        ibm_aug = add_cluster_one_hot(ibm_test_imp, ibm_test_cluster, k)

        aug_model = fit_xgb(train_aug, source_y_train)
        aug_table = evaluate_three_domains(
            aug_model,
            cell_aug,
            cy_test,
            bank_aug,
            by_test,
            ibm_aug,
            iy_test,
        )

        router = router_fit(source_X_train_imp, source_y_train, source_train_cluster)
        router_table = pd.DataFrame(
            [
                {"split": "Cell2Cell holdout", **compute_metrics(cy_test, router_predict(router, cell_test_imp, source_test_cluster.iloc[: len(cX_test)].reset_index(drop=True)))},
                {"split": "BankChurners holdout", **compute_metrics(by_test, router_predict(router, bank_test_imp, source_test_cluster.iloc[len(cX_test):].reset_index(drop=True)))},
                {"split": "IBM holdout", **compute_metrics(iy_test, router_predict(router, ibm_X_imp, ibm_test_cluster))},
            ]
        )

        sweep_rows.append(
            {
                "k": k,
                **diag,
                "aug_ibm_auc": float(aug_table.loc[aug_table["split"] == "IBM holdout", "roc_auc"].iloc[0]),
                "aug_ibm_f1": float(aug_table.loc[aug_table["split"] == "IBM holdout", "f1"].iloc[0]),
                "router_ibm_auc": float(router_table.loc[router_table["split"] == "IBM holdout", "roc_auc"].iloc[0]),
                "router_ibm_f1": float(router_table.loc[router_table["split"] == "IBM holdout", "f1"].iloc[0]),
            }
        )
        augmented_tables.append({"k": k, "table": aug_table})
        router_tables.append({"k": k, "table": router_table})

    sweep_df = pd.DataFrame(sweep_rows)
    best_aug_k = int(sweep_df.sort_values(["aug_ibm_auc", "aug_ibm_f1"], ascending=False).iloc[0]["k"])
    best_router_k = int(sweep_df.sort_values(["router_ibm_auc", "router_ibm_f1"], ascending=False).iloc[0]["k"])

    best_aug_table = next(item["table"] for item in augmented_tables if item["k"] == best_aug_k)
    best_router_table = next(item["table"] for item in router_tables if item["k"] == best_router_k)
    best_aug_cluster = cluster_tables[best_aug_k]
    best_router_cluster = cluster_tables[best_router_k]

    result_lines = []
    result_lines.append("# Cluster Router Experiment on abstract_shared without tenure")
    result_lines.append("")
    result_lines.append("## Goal")
    result_lines.append("- Check whether clustering on the shared abstract representation helps cross-domain churn transfer.")
    result_lines.append("- Source domains: Cell2Cell and BankChurners.")
    result_lines.append("- Target domain: IBM Telco.")
    result_lines.append("- Methods: baseline pooled XGBoost, cluster-augmented XGBoost, and cluster router (mixture of experts).")
    result_lines.append("")
    result_lines.append("## Baseline pooled model")
    result_lines.append(md_table(baseline_table.round(4)))
    result_lines.append("")
    result_lines.append("### IBM best-threshold check for baseline")
    result_lines.append(f"- best_threshold: {baseline_ibm_best[0]:.3f}")
    result_lines.append(f"- best_f1: {baseline_ibm_best[1]:.4f}")
    result_lines.append("")
    result_lines.append("## K sweep diagnostics")
    result_lines.append(md_table(sweep_df.round(4)))
    result_lines.append("")
    result_lines.append(f"## Best cluster-augmented model: k={best_aug_k}")
    result_lines.append(md_table(best_aug_table.round(4)))
    result_lines.append("")
    result_lines.append(f"## Best cluster-router model: k={best_router_k}")
    result_lines.append(md_table(best_router_table.round(4)))
    result_lines.append("")
    result_lines.append(f"## Cluster composition for best cluster-augmented k={best_aug_k}")
    result_lines.append(best_aug_cluster["source"].round(4).to_string(index=False))
    result_lines.append("")
    result_lines.append(f"### IBM cluster composition for best cluster-augmented k={best_aug_k}")
    result_lines.append(best_aug_cluster["ibm"].round(4).to_string(index=False))
    result_lines.append("")
    result_lines.append(f"## Cluster composition for best router k={best_router_k}")
    result_lines.append(best_router_cluster["source"].round(4).to_string(index=False))
    result_lines.append("")
    result_lines.append(f"### IBM cluster composition for best router k={best_router_k}")
    result_lines.append(best_router_cluster["ibm"].round(4).to_string(index=False))
    result_lines.append("")
    result_lines.append("## Interpretation")
    result_lines.append("- If cluster purity is high but IBM AUC does not improve, clustering is mostly acting as a domain separator rather than a shared representation learner.")
    result_lines.append("- If the router beats the baseline on IBM, then cluster-based routing is a viable way to handle the domain mismatch.")
    result_lines.append("- If neither the augmented model nor the router improves IBM, then clustering is not solving the meaning drift across domains.")
    result_text = "\n".join(result_lines)
    RESULT_PATH.write_text(result_text, encoding="utf-8")

    print("Wrote:", RESULT_PATH)
    print()
    print("=== Baseline pooled model ===")
    print(md_table(baseline_table.round(4)))
    print()
    print("=== K sweep diagnostics ===")
    print(md_table(sweep_df.round(4)))
    print()
    print(f"=== Best cluster-augmented model: k={best_aug_k} ===")
    print(md_table(best_aug_table.round(4)))
    print()
    print(f"=== Best cluster-router model: k={best_router_k} ===")
    print(md_table(best_router_table.round(4)))

    script_path = Path(__file__) if "__file__" in globals() else None
    if script_path is not None and script_path.exists():
        write_notebook(script_path.read_text(encoding="utf-8"))


if __name__ == "__main__":
    main()
